# t-SNE analysis of real and synthetic PneumoniaMNIST images

This notebook reproduces both t-SNE analyses reported in the project: one using an ImageNet-pretrained ResNet-18 feature space and one using the independently trained PneumoniaMNIST ResNet-18 feature space. Four groups of 400 images are sampled with seed 42. PCA reduces the 512-dimensional features to 50 dimensions before t-SNE.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
!pip install -q medmnist scikit-learn pandas matplotlib tqdm pillow


## A. ImageNet-pretrained ResNet-18 features


In [ ]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset

from torchvision import models, transforms
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

import medmnist
from medmnist import PneumoniaMNIST

# Fix the seed and balance all four source/class groups.
SEED = 42
N_PER_GROUP = 400

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

In [ ]:
from pathlib import Path

SYNTH_NORMAL_DIR = Path(
    "/content/drive/MyDrive/MedSymmFlow_Project/"
    "synthetic_data/pneumoniamnist_normal_3200_raw"
)

SYNTH_PNEUMONIA_DIR = Path(
    "/content/drive/MyDrive/MedSymmFlow_Project/"
    "synthetic_data/pneumoniamnist_pneumonia_400_raw"
)

synthetic_normal_paths = sorted(
    SYNTH_NORMAL_DIR.rglob("*.png")
)

synthetic_pneumonia_paths = sorted(
    SYNTH_PNEUMONIA_DIR.rglob("*.png")
)

print("Synthetic Normal:", len(synthetic_normal_paths))
print("Synthetic Pneumonia:", len(synthetic_pneumonia_paths))

print("\nNormal root:")
print(SYNTH_NORMAL_DIR)

print("\nPneumonia root:")
print(SYNTH_PNEUMONIA_DIR)

In [ ]:
from medmnist import PneumoniaMNIST
import numpy as np

SEED = 42
N_PER_GROUP = 400

# Load original 28x28 training set
real_dataset = PneumoniaMNIST(
    split="train",
    download=True,
    size=28
)

real_normal_indices = []
real_pneumonia_indices = []

for idx in range(len(real_dataset)):
    _, label = real_dataset[idx]
    label = int(np.asarray(label).squeeze())

    if label == 0:
        real_normal_indices.append(idx)
    elif label == 1:
        real_pneumonia_indices.append(idx)

print("Full real training set:")
print("Real Normal:", len(real_normal_indices))
print("Real Pneumonia:", len(real_pneumonia_indices))

# Fixed random sampling
rng = np.random.default_rng(SEED)

selected_real_normal = rng.choice(
    real_normal_indices,
    size=N_PER_GROUP,
    replace=False
)

selected_real_pneumonia = rng.choice(
    real_pneumonia_indices,
    size=N_PER_GROUP,
    replace=False
)

selected_synth_normal = rng.choice(
    len(synthetic_normal_paths),
    size=N_PER_GROUP,
    replace=False
)

selected_synth_pneumonia = rng.choice(
    len(synthetic_pneumonia_paths),
    size=N_PER_GROUP,
    replace=False
)

selected_synth_normal_paths = [
    synthetic_normal_paths[i]
    for i in selected_synth_normal
]

selected_synth_pneumonia_paths = [
    synthetic_pneumonia_paths[i]
    for i in selected_synth_pneumonia
]

print("\nSelected for t-SNE:")
print("Real Normal:", len(selected_real_normal))
print("Real Pneumonia:", len(selected_real_pneumonia))
print("Synthetic Normal:", len(selected_synth_normal_paths))
print("Synthetic Pneumonia:", len(selected_synth_pneumonia_paths))
print("Total:", 4 * N_PER_GROUP)

In [ ]:
import torch
import torch.nn as nn
import numpy as np

from PIL import Image
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

# --------------------------------------------------
# ResNet-18 pretrained on ImageNet
# --------------------------------------------------

weights = models.ResNet18_Weights.IMAGENET1K_V1

resnet = models.resnet18(
    weights=weights
)

# Remove final fully-connected classification layer.
# Output will be a 512-dimensional feature vector.
feature_extractor = nn.Sequential(
    *list(resnet.children())[:-1]
).to(device)

feature_extractor.eval()

# PneumoniaMNIST is grayscale 28x28.
# Convert to 3 channels and resize for ImageNet ResNet-18.
feature_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# --------------------------------------------------
# Dataset containing all four groups
# --------------------------------------------------

class TSNEImageDataset(Dataset):

    def __init__(
        self,
        real_dataset,
        real_normal_indices,
        real_pneumonia_indices,
        synth_normal_paths,
        synth_pneumonia_paths,
        transform
    ):
        self.samples = []
        self.real_dataset = real_dataset
        self.transform = transform

        for idx in real_normal_indices:
            self.samples.append(
                ("real", int(idx), "Real Normal")
            )

        for idx in real_pneumonia_indices:
            self.samples.append(
                ("real", int(idx), "Real Pneumonia")
            )

        for path in synth_normal_paths:
            self.samples.append(
                ("synthetic", path, "Synthetic Normal")
            )

        for path in synth_pneumonia_paths:
            self.samples.append(
                ("synthetic", path, "Synthetic Pneumonia")
            )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        source, item, group = self.samples[idx]

        if source == "real":
            image, _ = self.real_dataset[item]

            if not isinstance(image, Image.Image):
                image = Image.fromarray(
                    np.asarray(image).squeeze().astype(np.uint8)
                )

        else:
            image = Image.open(item).convert("L")

        image = self.transform(image)

        return image, group


tsne_dataset = TSNEImageDataset(
    real_dataset=real_dataset,
    real_normal_indices=selected_real_normal,
    real_pneumonia_indices=selected_real_pneumonia,
    synth_normal_paths=selected_synth_normal_paths,
    synth_pneumonia_paths=selected_synth_pneumonia_paths,
    transform=feature_transform
)

tsne_loader = DataLoader(
    tsne_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2
)

print("Images in t-SNE dataset:", len(tsne_dataset))


# --------------------------------------------------
# Extract 512-dimensional features
# --------------------------------------------------

all_features = []
all_groups = []

with torch.no_grad():

    for images, groups in tsne_loader:

        images = images.to(device)

        features = feature_extractor(images)

        features = features.flatten(1)

        all_features.append(
            features.cpu().numpy()
        )

        all_groups.extend(groups)


all_features = np.concatenate(
    all_features,
    axis=0
)

all_groups = np.asarray(all_groups)

print("\nFeature extraction complete.")
print("Feature matrix shape:", all_features.shape)

unique, counts = np.unique(
    all_groups,
    return_counts=True
)

print("\nGroups:")
for group, count in zip(unique, counts):
    print(group, ":", count)

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import pandas as pd
import numpy as np

SEED = 42

# --------------------------------------------------
# 1. PCA: 512 -> 50 dimensions
# --------------------------------------------------

pca = PCA(
    n_components=50,
    random_state=SEED
)

features_pca = pca.fit_transform(
    all_features
)

print("After PCA:", features_pca.shape)

print(
    "Explained variance:",
    round(
        pca.explained_variance_ratio_.sum(),
        4
    )
)

# --------------------------------------------------
# 2. t-SNE: 50 -> 2 dimensions
# --------------------------------------------------

tsne = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate="auto",
    init="pca",
    max_iter=1500,
    random_state=SEED
)

tsne_embedding = tsne.fit_transform(
    features_pca
)

print(
    "t-SNE embedding:",
    tsne_embedding.shape
)

# --------------------------------------------------
# 3. Prepare dataframe
# --------------------------------------------------

tsne_df = pd.DataFrame({
    "tsne_1": tsne_embedding[:, 0],
    "tsne_2": tsne_embedding[:, 1],
    "group": all_groups
})

display(
    tsne_df.groupby("group").size()
)

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/MedSymmFlow_Project/tsne_analysis"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Encode diagnosis by color and data source by marker shape.
styles = {
    "Real Normal": {
        "color": "tab:blue",
        "marker": "o"
    },
    "Synthetic Normal": {
        "color": "tab:blue",
        "marker": "x"
    },
    "Real Pneumonia": {
        "color": "tab:red",
        "marker": "o"
    },
    "Synthetic Pneumonia": {
        "color": "tab:red",
        "marker": "x"
    }
}

plt.figure(figsize=(10, 8))

for group, style in styles.items():

    mask = tsne_df["group"] == group

    plt.scatter(
        tsne_df.loc[mask, "tsne_1"],
        tsne_df.loc[mask, "tsne_2"],
        label=group,
        color=style["color"],
        marker=style["marker"],
        s=32,
        alpha=0.65
    )

plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")

plt.title(
    "t-SNE of Real and Synthetic PneumoniaMNIST Images\n"
    "ResNet-18 ImageNet Features"
)

plt.legend(
    frameon=True,
    fontsize=10
)

plt.tight_layout()

SAVE_PATH = OUTPUT_DIR / "tsne_real_vs_synthetic.png"

plt.savefig(
    SAVE_PATH,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved to:")
print(SAVE_PATH)

In [ ]:
CSV_PATH = OUTPUT_DIR / "tsne_real_vs_synthetic_coordinates.csv"

tsne_df.to_csv(
    CSV_PATH,
    index=False
)

print("Coordinates saved to:")
print(CSV_PATH)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Downsample only for visual clarity; t-SNE was fitted on all selected images.
PLOT_SEED = 42
N_SHOW = 250

rng_plot = np.random.default_rng(PLOT_SEED)

plot_indices = []

for group in [
    "Real Normal",
    "Synthetic Normal",
    "Real Pneumonia",
    "Synthetic Pneumonia"
]:
    group_idx = np.where(tsne_df["group"].values == group)[0]

    selected_idx = rng_plot.choice(
        group_idx,
        size=N_SHOW,
        replace=False
    )

    plot_indices.extend(selected_idx)

plot_df = tsne_df.iloc[plot_indices].copy()

styles = {
    "Real Normal": {
        "color": "tab:blue",
        "marker": "o"
    },
    "Synthetic Normal": {
        "color": "tab:blue",
        "marker": "x"
    },
    "Real Pneumonia": {
        "color": "tab:red",
        "marker": "o"
    },
    "Synthetic Pneumonia": {
        "color": "tab:red",
        "marker": "x"
    }
}

plt.figure(figsize=(10, 8))

for group, style in styles.items():
    mask = plot_df["group"] == group

    plt.scatter(
        plot_df.loc[mask, "tsne_1"],
        plot_df.loc[mask, "tsne_2"],
        label=group,
        marker=style["marker"],
        s=18,
        alpha=0.45,
        linewidths=0.8
    )

plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")

plt.title(
    "t-SNE of Real and Synthetic PneumoniaMNIST Images\n"
    "250 samples per group"
)

plt.legend(
    frameon=True,
    fontsize=10
)

plt.tight_layout()

CLEAN_PATH = (
    OUTPUT_DIR
    / "tsne_real_vs_synthetic_clean.png"
)

plt.savefig(
    CLEAN_PATH,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved to:")
print(CLEAN_PATH)

In [ ]:
# --------------------------------------------------
# NORMAL: Real vs Synthetic
# --------------------------------------------------

plt.figure(figsize=(8, 7))

for group, marker in [
    ("Real Normal", "o"),
    ("Synthetic Normal", "x")
]:
    mask = plot_df["group"] == group

    plt.scatter(
        plot_df.loc[mask, "tsne_1"],
        plot_df.loc[mask, "tsne_2"],
        label=group,
        marker=marker,
        s=20,
        alpha=0.5
    )

plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.title("Normal: Real vs Synthetic")
plt.legend()
plt.tight_layout()

NORMAL_PATH = (
    OUTPUT_DIR
    / "tsne_normal_real_vs_synthetic.png"
)

plt.savefig(
    NORMAL_PATH,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# --------------------------------------------------
# PNEUMONIA: Real vs Synthetic
# --------------------------------------------------

plt.figure(figsize=(8, 7))

for group, marker in [
    ("Real Pneumonia", "o"),
    ("Synthetic Pneumonia", "x")
]:
    mask = plot_df["group"] == group

    plt.scatter(
        plot_df.loc[mask, "tsne_1"],
        plot_df.loc[mask, "tsne_2"],
        label=group,
        marker=marker,
        s=20,
        alpha=0.5
    )

plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.title("Pneumonia: Real vs Synthetic")
plt.legend()
plt.tight_layout()

PNEUMONIA_PATH = (
    OUTPUT_DIR
    / "tsne_pneumonia_real_vs_synthetic.png"
)

plt.savefig(
    PNEUMONIA_PATH,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved:")
print(NORMAL_PATH)
print(PNEUMONIA_PATH)

t-SNE visualization of ResNet-18 ImageNet feature embeddings for real and synthetic PneumoniaMNIST images. Real and synthetic samples show substantial overlap within each class, while synthetic samples occupy a somewhat narrower and partially shifted region of the feature space.

## B. PneumoniaMNIST-trained ResNet-18 features

This section loads `best_pneumoniamnist_resnet18.pth` from the project Drive folder and uses the penultimate 512-dimensional layer as the feature representation.


In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from medmnist import PneumoniaMNIST

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

import matplotlib.pyplot as plt
from tqdm import tqdm

In [ ]:
PROJECT_ROOT = Path("/content/drive/MyDrive/MedSymmFlow_Project")
SYNTH_ROOT = PROJECT_ROOT / "synthetic_data"
TSNE_OUT = PROJECT_ROOT / "tsne_analysis_trained_resnet"
TSNE_OUT.mkdir(parents=True, exist_ok=True)

print("Project exists:", PROJECT_ROOT.exists())
print("Synthetic root exists:", SYNTH_ROOT.exists())
print("Output dir:", TSNE_OUT)

In [ ]:
SYNTH_NORMAL_ROOT = SYNTH_ROOT / "pneumoniamnist_normal_3200_raw"
SYNTH_PNEUMONIA_ROOT = SYNTH_ROOT / "pneumoniamnist_pneumonia_400_raw"

print("Normal root:", SYNTH_NORMAL_ROOT)
print("Exists:", SYNTH_NORMAL_ROOT.exists())

print("Pneumonia root:", SYNTH_PNEUMONIA_ROOT)
print("Exists:", SYNTH_PNEUMONIA_ROOT.exists())

In [ ]:
def collect_pngs(root):
    return sorted(root.rglob("*.png"))

synthetic_normal_paths = collect_pngs(SYNTH_NORMAL_ROOT)
synthetic_pneumonia_paths = collect_pngs(SYNTH_PNEUMONIA_ROOT)

print("Synthetic Normal:", len(synthetic_normal_paths))
print("Synthetic Pneumonia:", len(synthetic_pneumonia_paths))

In [ ]:
real_train = PneumoniaMNIST(split="train", download=True, as_rgb=True)
print("Full real training set:", len(real_train))

In [ ]:
real_normal_indices = []
real_pneumonia_indices = []

for i in range(len(real_train)):
    _, label = real_train[i]
    y = int(label[0])
    if y == 0:
        real_normal_indices.append(i)
    else:
        real_pneumonia_indices.append(i)

print("Real Normal:", len(real_normal_indices))
print("Real Pneumonia:", len(real_pneumonia_indices))

In [ ]:
# Reuse a fixed, balanced sample for the task-specific feature analysis.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

N_PER_GROUP = 400

selected_real_normal_idx = random.sample(real_normal_indices, N_PER_GROUP)
selected_real_pneumonia_idx = random.sample(real_pneumonia_indices, N_PER_GROUP)
selected_synth_normal_paths = random.sample(synthetic_normal_paths, N_PER_GROUP)
selected_synth_pneumonia_paths = random.sample(synthetic_pneumonia_paths, N_PER_GROUP)

print("Selected for t-SNE:")
print("Real Normal:", len(selected_real_normal_idx))
print("Real Pneumonia:", len(selected_real_pneumonia_idx))
print("Synthetic Normal:", len(selected_synth_normal_paths))
print("Synthetic Pneumonia:", len(selected_synth_pneumonia_paths))
print("Total:", 4 * N_PER_GROUP)

In [ ]:
class TSNEDataset(Dataset):
    def __init__(self,
                 real_dataset,
                 real_normal_idx,
                 real_pneumonia_idx,
                 synth_normal_paths,
                 synth_pneumonia_paths,
                 transform=None):
        self.samples = []
        self.transform = transform

        for idx in real_normal_idx:
            self.samples.append(("real", "normal", idx))

        for idx in real_pneumonia_idx:
            self.samples.append(("real", "pneumonia", idx))

        for path in synth_normal_paths:
            self.samples.append(("synthetic", "normal", path))

        for path in synth_pneumonia_paths:
            self.samples.append(("synthetic", "pneumonia", path))

        self.real_dataset = real_dataset

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        source, label_name, data_ref = self.samples[i]

        if source == "real":
            img, _ = self.real_dataset[data_ref]
        else:
            img = Image.open(data_ref).convert("RGB")

        if self.transform is not None:
            img = self.transform(img)

        group = f"{source.capitalize()} {label_name.capitalize()}"
        label_binary = 0 if label_name == "normal" else 1

        return img, label_binary, group

In [ ]:
from pathlib import Path

# Load the independently trained classifier used in downstream experiments.
CHECKPOINT_PATH = Path(
    "/content/drive/MyDrive/MedSymmFlow_Project/"
    "best_pneumoniamnist_resnet18.pth"
)

print("Checkpoint:")
print(CHECKPOINT_PATH)
print("Exists:", CHECKPOINT_PATH.exists())
print("Size (MB):", round(CHECKPOINT_PATH.stat().st_size / 1024**2, 2))

In [ ]:
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=False
)

print("Checkpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("\nCheckpoint keys:")
    for key in checkpoint.keys():
        print(key)

In [ ]:
from torchvision import models
import torch.nn as nn

model = models.resnet18(weights=None)

model.fc = nn.Linear(
    model.fc.in_features,
    2
)

state_dict = checkpoint["model_state"]

# Remove a DataParallel prefix if present
state_dict = {
    key.replace("module.", ""): value
    for key, value in state_dict.items()
}

model.load_state_dict(state_dict)

model = model.to(device)
model.eval()

print("Checkpoint loaded successfully.")
print("Best epoch:", checkpoint["epoch"])
print("Validation AUC:", checkpoint["val_auc"])

In [ ]:
from torchvision import transforms

inference_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
tsne_dataset = TSNEDataset(
    real_dataset=real_train,
    real_normal_idx=selected_real_normal_idx,
    real_pneumonia_idx=selected_real_pneumonia_idx,
    synth_normal_paths=selected_synth_normal_paths,
    synth_pneumonia_paths=selected_synth_pneumonia_paths,
    transform=inference_transform
)

tsne_loader = DataLoader(
    tsne_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Images in t-SNE dataset:", len(tsne_dataset))

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm

# Remove the classification head to obtain task-specific feature vectors.
feature_extractor = nn.Sequential(
    *list(model.children())[:-1]
).to(device)

feature_extractor.eval()

all_features_trained = []
all_groups_trained = []

with torch.no_grad():
    for images, labels, groups in tqdm(tsne_loader):

        images = images.to(
            device,
            non_blocking=True
        )

        features = feature_extractor(images)

        features = features.flatten(1)

        all_features_trained.append(
            features.cpu().numpy()
        )

        all_groups_trained.extend(groups)

all_features_trained = np.concatenate(
    all_features_trained,
    axis=0
)

all_groups_trained = np.asarray(
    all_groups_trained
)

print("\nFeature extraction complete.")
print(
    "Feature matrix shape:",
    all_features_trained.shape
)

print("\nGroups:")
print(
    pd.Series(
        all_groups_trained
    ).value_counts()
)

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import pandas as pd

SEED = 42

# Reduce dimensionality before t-SNE to suppress noise and lower runtime.
# PCA: 512 -> 50
pca_trained = PCA(
    n_components=50,
    random_state=SEED
)

features_pca_trained = pca_trained.fit_transform(
    all_features_trained
)

print("After PCA:", features_pca_trained.shape)
print(
    "Explained variance:",
    round(pca_trained.explained_variance_ratio_.sum(), 4)
)

# t-SNE: 50 -> 2
tsne_trained = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate="auto",
    init="pca",
    max_iter=1500,
    random_state=SEED
)

embedding_trained = tsne_trained.fit_transform(
    features_pca_trained
)

print("t-SNE embedding:", embedding_trained.shape)

tsne_trained_df = pd.DataFrame({
    "tsne_1": embedding_trained[:, 0],
    "tsne_2": embedding_trained[:, 1],
    "group": all_groups_trained
})

print("\nGroups:")
print(tsne_trained_df["group"].value_counts())

In [ ]:
import matplotlib.pyplot as plt

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/MedSymmFlow_Project/tsne_analysis_trained_resnet"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Normal
normal_df = tsne_trained_df[
    tsne_trained_df["group"].isin(
        ["Real Normal", "Synthetic Normal"]
    )
]

plt.figure(figsize=(8, 7))

for group, marker in [
    ("Real Normal", "o"),
    ("Synthetic Normal", "x")
]:
    mask = normal_df["group"] == group

    plt.scatter(
        normal_df.loc[mask, "tsne_1"],
        normal_df.loc[mask, "tsne_2"],
        label=group,
        marker=marker,
        s=24,
        alpha=0.6
    )

plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.title(
    "Normal: Real vs Synthetic\n"
    "Features from PneumoniaMNIST-trained ResNet-18"
)
plt.legend()
plt.tight_layout()

normal_path = OUTPUT_DIR / "tsne_trained_resnet_normal.png"
plt.savefig(normal_path, dpi=300, bbox_inches="tight")
plt.show()


# Pneumonia
pneumonia_df = tsne_trained_df[
    tsne_trained_df["group"].isin(
        ["Real Pneumonia", "Synthetic Pneumonia"]
    )
]

plt.figure(figsize=(8, 7))

for group, marker in [
    ("Real Pneumonia", "o"),
    ("Synthetic Pneumonia", "x")
]:
    mask = pneumonia_df["group"] == group

    plt.scatter(
        pneumonia_df.loc[mask, "tsne_1"],
        pneumonia_df.loc[mask, "tsne_2"],
        label=group,
        marker=marker,
        s=24,
        alpha=0.6
    )

plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.title(
    "Pneumonia: Real vs Synthetic\n"
    "Features from PneumoniaMNIST-trained ResNet-18"
)
plt.legend()
plt.tight_layout()

pneumonia_path = OUTPUT_DIR / "tsne_trained_resnet_pneumonia.png"
plt.savefig(pneumonia_path, dpi=300, bbox_inches="tight")
plt.show()

print(normal_path)
print(pneumonia_path)

To assess the similarity between real and synthetic images at the level of the classifier’s internal feature representation, 400 images were sampled from each of four groups: Real Normal, Synthetic Normal, Real Pneumonia, and Synthetic Pneumonia. All images were passed through a ResNet-18 classifier trained only on real PneumoniaMNIST images, and 512-dimensional feature vectors were extracted from the layer immediately before the final classification layer. These feature vectors were first reduced to 50 dimensions using PCA, which preserved 92.42% of the total variance, and were then projected into two dimensions using t-SNE. For clearer interpretation, two separate plots were generated, one for the Normal class and one for the Pneumonia class, each comparing real and synthetic samples. In both plots, partial overlap was observed between real and synthetic images, but a clear visual separation between their feature distributions was also evident. These findings suggest that the synthetic images capture class-relevant characteristics, but do not fully reproduce the feature distribution of the real data. This indicates a certain degree of distribution shift between real and synthetic samples and may help explain why synthetic augmentation can provide limited improvement, while not serving as a complete substitute for real training data.